# Clustering and Annotation

We performed data normalization, dimensionality reduction, and batch correction  
to eliminate technical variability among samples, followed by clustering and cell-type annotation  
to identify distinct cellular populations.  
All processes were conducted using the **OmicVerse** framework in *Python*.

In [ ]:
import scanpy as sc
import omicverse as ov
import numpy as np
import celltypist
from celltypist import models
import seaborn as sns
import matplotlib.pyplot as plt
ov.plot_set(dpi=150,font_path='Arial')

# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

In [ ]:
ov.settings.cpu_init()

In [ ]:
# AnnData Loading
adata_raw = sc.read_h5ad(r"F:\Data\snRNAseq\PlaceCell\FilteredData\PCAnalysis_all_QC_20260127.h5ad")
adata_raw

In [ ]:
# adata_annota = sc.read_h5ad(r"F:\Data\snRNAseq\BGI\analysis-Min Hu\FilteredData\SoupX\20251023\All_QC_filtered_annotation_nobg_20251121.h5ad")
# adata_annota

In [ ]:
# cell_to_keep = adata_annota.obs_names
# adata = adata_raw[cell_to_keep, :].copy()
# adata

In [ ]:
adata = adata_raw.copy()

In [ ]:
# Perform mild quality control (QC)
adata = ov.pp.qc(
    adata,
    mt_startswith='mt-',              # Automatically detect mitochondrial genes (prefix: 'mt-')
    tresh={
        'mito_perc': 0.05,            # Filter cells with mitochondrial percentage < 5%
        'nUMIs': 1000,                 # Keep cells with total counts ≥ 800
        'detected_genes': 400         # Keep cells with ≥ 200 detected genes
    },
    # mode = 'mads',
    min_cells=3,                     # Retain genes expressed in at least 10 cells
    min_genes=200,                    # Retain cells expressing at least 200 genes
    # batch_key='sample',
    doublets=False,                   # Disable doublet detection (optional)
)


In [ ]:
adata

In [ ]:
sc.pl.violin(
    adata,
    ["detected_genes", "nUMIs", "mito_perc"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
adata_raw = adata.copy()
adata_raw.layers["counts"] = adata_raw.X.copy()  # 永远保留原始 counts

---

### Statistical Analysis: Detected Genes per Sample by Condition

We compared the number of detected genes across individual mice, grouped by experimental condition.
Each point represents a single cell; the boxplot summarizes the distribution for each sample.
Statistical significance between conditions was assessed using the Mann-Whitney U test.

---


---

### 1. Highly Variable Gene Detection

We identified highly variable genes (HVGs) using **Pearson’s method**,  
which has been shown to outperform conventional normalization approaches.

---


In [ ]:
adata=ov.pp.preprocess(adata,mode='shiftlog|pearson',n_HVGs=2000,organism='mouse',
                       target_sum=50*1e4, batch_key='sample')
adata

In [ ]:
adata

In [ ]:
adata_hvg = adata[:, adata.var.highly_variable_features]
adata_hvg

---

### 2. Normalization and primary dimensionality reduction

Dimensionality reduction embeds high-dimensional data into a lower-dimensional space.  
The reduced representation preserves the main sources of biological variability  
and facilitates downstream analyses such as clustering and visualization.

---


In [ ]:
ov.pp.scale(adata_hvg)
adata_hvg

In [ ]:
ov.pp.pca(adata_hvg,layer='scaled',n_pcs=50)
adata_hvg

In [ ]:
adata_hvg.obsm['X_pca']=adata_hvg.obsm['scaled|original|X_pca']
ov.pl.embedding(adata_hvg,
                  basis='X_pca',
                  color='sample',
                  frameon='small')

---

### 3. Batch Correction

We used **Harmony** to correct batch effects across samples.  
**Harmony** is an algorithm designed for the integration of single-cell genomics datasets.  

---


In [ ]:
ov.single.batch_correction(adata_hvg,
                           batch_key='sample',
                           methods='harmony',
                           n_pcs=50)
adata_hvg

---

### 4. Embedding the neighborhood graph

To visualize the PCA embeddings, we applied the pymde implementation integrated in OmicVerse.
This method serves as a GPU-accelerated alternative to UMAP, providing smoother and more stable visualizations of the neighborhood structure.

---


In [ ]:
ov.pp.mde(adata_hvg,embedding_dim=2,n_neighbors=15, basis='X_mde',
          n_pcs=30, metric='euclidean',repulsive_fraction=0.9, use_rep='X_pca_harmony',random_state=123)

In [ ]:
adata

In [ ]:
X = adata_hvg.obsm["X_mde"]

x = X[:,0]
y = X[:,1]

X_rot = np.column_stack([y, x])

adata_hvg.obsm["X_mde"] = X_rot

In [ ]:
adata

In [ ]:
fig = ov.pl.embedding(
    adata_hvg,
    basis='X_mde',
    color=['sample'],
    frameon='small',
    show=False,
    return_fig=True
)

fig.savefig("embedding_sample.svg", bbox_inches="tight")
fig.savefig("embedding_sample.png", dpi=300, bbox_inches="tight")

In [ ]:
# 1) sync obs
adata.obs = adata_hvg.obs.copy()

# 2) sync obsm
for key in adata_hvg.obsm.keys():
    adata.obsm[key] = adata_hvg.obsm[key].copy()

# 3) sync uns（所有非基因相关的 uns 都可以同步）
for key in adata_hvg.uns.keys():
    # 跳过与 var /基因相关的 uns
    if key not in ["hvg", "pca", "scaled|original|pca_var_ratios", "scaled|original|cum_sum_eigenvalues"]:
        adata.uns[key] = adata_hvg.uns[key].copy()

# 4) sync neighbor graph (非常重要)
adata.obsp["distances"] = adata_hvg.obsp["distances"].copy()
adata.obsp["connectivities"] = adata_hvg.obsp["connectivities"].copy()

# 5) sync neighbors metadata
adata.uns["neighbors"] = adata_hvg.uns["neighbors"].copy()

print("obs, obsm, uns, obsp successfully synced!")


In [ ]:
marker_genes = {

    "Neuron": ['Rbfox3','Meg3','Syt1','Snap25'],
    # 兴奋性神经元（Cortical excitatory neurons）
    "Excitatory Neuron":     ["Slc17a7", "Sv2b", "Neurod2"],

    # 抑制性神经元（GABAergic interneurons）
    "GABAergic Neuron":      ["Gad1", "Gad2", "Slc32a1"],

    # 星形胶质细胞（Astrocytes）
    # "Astrocyte":                   ["Gja1", "Aldoc", "Mfge8","Cldn10","Aqp4","Acsbg1","Gfap"],

    # # 少突胶质细胞及前体（Oligodendrocyte lineage）
    # "OPC":                         ["Pdgfra", "Cspg4", "C1ql1","Tnfaip2"],
    # "Oligodendrocyte":             ["Cldn11",     "Plp1",    "Mog"],

    # # 小胶质细胞（Microglia）
    # "Microglia":                   ["Cx3cr1",  "C1qa",  "C1qb", "C1qc", "Tmem119", "Ctss"],

    # # 血管相关细胞（Vascular cells）
    # "Endothelial":                 ["Rgs5",  "Cldn5", "Ly6a"],

    # # 其它（可按需添加）
    # "Ependymal":                   ["Folr1",   "Kcnj13",  "Sostdc1"],
}


In [ ]:
from matplotlib.colors import LinearSegmentedColormap
colors = ["#22456F", "#528FAC","#AECAD8","#F2F2F2","#F7D29D","#EEB84B", "#E87A2D"]  # 示例颜色
cmap_custom = LinearSegmentedColormap.from_list("custom_cmap", colors, N=256)

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# SVG 文字在 AI 中可编辑
mpl.rcParams["svg.fonttype"] = "none"

# =========================
# 1. sample 离散配色
# =========================
sample_key = "sample"

# 推荐：Tableau 10 风格，适合 10 个离散 sample
sample_colors = [
    "#4E79A7",  # blue
    "#F28E2B",  # orange
    "#59A14F",  # green
    "#E15759",  # red
    "#B07AA1",  # purple
    "#9C755F",  # brown
    "#EDC948",  # yellow
    "#76B7B2",  # cyan
    "#FF9DA7",  # pink
    "#79706E"   # gray
]

# 按 PC 数字自然排序，例如 PC92, PC94, PC103...
def natural_key(x):
    m = re.search(r"\d+", str(x))
    return int(m.group()) if m else str(x)

sample_order = sorted(
    adata_hvg.obs[sample_key].dropna().astype(str).unique(),
    key=natural_key
)

if len(sample_order) > len(sample_colors):
    raise ValueError(
        f"sample 数量为 {len(sample_order)}，但 sample_colors 只有 {len(sample_colors)} 个，请增加颜色。"
    )

sample_palette = {
    sample: sample_colors[i]
    for i, sample in enumerate(sample_order)
}

adata_hvg.obs[sample_key] = pd.Categorical(
    adata_hvg.obs[sample_key].astype(str),
    categories=sample_order,
    ordered=True
)

print("Sample color mapping:")
for k, v in sample_palette.items():
    print(f"{k}: {v}")

# =========================
# 2. 绘图：保持你之前的设置
# =========================
plt.figure(figsize=(10, 9))

name = "sample"

fig = ov.pl.embedding(
    adata_hvg,
    basis="X_mde",
    color=[sample_key],
    palette=sample_palette,
    size=100,                    # 保持不变
    alpha=0.7,                   # 保持不变
    frameon="small",
    legend_loc="right margin",
    legend_fontsize=10,
    legend_fontweight="normal",
    add_outline=False,
    show=False,
    return_fig=True
)

# =========================
# 3. 保存
# =========================
save_dir = r"F:\Figures\PlaceCells\Figure3"
os.makedirs(save_dir, exist_ok=True)

fig.savefig(f"{save_dir}/{name}.svg", bbox_inches="tight")
fig.savefig(f"{save_dir}/{name}.png", dpi=600, bbox_inches="tight")

plt.show()
plt.close(fig)

In [ ]:
import os

plt.figure(figsize=(10, 9))

palette = {
    "nonplace": "#E2A528",
    "place": "#6FA3C6",
    "silent": "#7F7F7F"
}
name = "condition"
fig = ov.pl.embedding(
    adata_hvg,
    basis="X_mde",
    color=['condition'],
    palette=palette,
    color_map = cmap_custom,
    size=100,
    alpha=0.7,
    frameon="small",
    legend_loc="right margin",
    legend_fontsize=10,
    legend_fontweight="normal",
    add_outline=False,
    show=False,
    return_fig=True,
    vmax=6.0
)

save_dir = r"F:\Figures\PlaceCells\Figure3"
os.makedirs(save_dir, exist_ok=True)

# fig.savefig(f"{save_dir}/{name}.svg", bbox_inches="tight")
# fig.savefig(f"{save_dir}/{name}.png", dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
ov.pl.embedding(adata,
                basis='X_mde',
                # color=marker_genes['Neuron'],
                color = ['Grik4','Htr1a'],
                color_map= cmap_custom,
                frameon='small')

In [ ]:
adata

In [ ]:
ov.pp.leiden(adata,resolution=1.0)

In [ ]:
ov.pl.embedding(adata,
                basis='X_mde',
                color=['leiden'],
                size=50,
                frameon='small')

In [ ]:
adata

In [ ]:
ov.pl.violin(
    adata,
    keys = 'detected_genes',
    groupby = 'leiden',
    figsize = (10,5)
)

In [ ]:
fig,ax=plt.subplots(figsize = (3,4))
ov.pl.cellproportion(adata=adata,celltype_clusters='leiden',
                    groupby='condition',legend=True,ax=ax)

In [ ]:
adata

In [ ]:
from matplotlib import patheffects
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4,4))

ov.pl.embedding(adata,
                  basis='X_mde',
                  color=['leiden'],
                   show=False, legend_loc=None, add_outline=False, 
                   frameon='small',legend_fontoutline=2,ax=ax
                 )

ov.utils.gen_mpl_labels(
    adata,
    'leiden',
    exclude=("None",),  
    basis='X_mde',
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize= 12 ,weight='bold',
                     path_effects=[patheffects.withStroke(linewidth=2, foreground='w')] ),
)


In [ ]:
ov.pl.dotplot(adata, 'Stmn4', groupby='condition',
             standard_scale='var')

In [ ]:
adata

In [ ]:
sc.tl.dendrogram(adata,'leiden',use_rep='X_pca_harmony')
sc.tl.rank_genes_groups(adata, 'leiden', use_rep='X_pca_harmony',
                        method='t-test',use_raw=False,key_added='leiden_ttest')
ov.pl.rank_genes_groups_dotplot(adata,groupby='leiden',
                                cmap='Spectral_r',key='leiden_ttest',
                                standard_scale='var',n_genes=30,dendrogram=False)

---

### 5. Automated Annotation

We performed automated cell type annotation using CellTypist, which provides rapid and accurate classification of single cells based on a curated reference model.

---


In [ ]:
adata_raw

In [ ]:
adata_annota = adata_raw.copy()
sc.pp.normalize_total(adata_annota, target_sum=1e4)
sc.pp.log1p(adata_annota)

In [ ]:
adata_annota.X.sum(axis = 1)

In [ ]:
# models.models_description()

In [ ]:
model = models.Model.load(model='Mouse_Isocortex_Hippocampus.pkl')
model

In [ ]:
# model.cell_types

In [ ]:
adata_annota

In [ ]:
predictions = celltypist.annotate(adata_annota, model = model, majority_voting = True, mode = 'prob match', p_thres = 0.3)

In [ ]:
predictions
adata_new = predictions.to_adata(insert_prob=True)

In [ ]:

adata_new.obsm['X_mde'] = adata_hvg.obsm['X_mde']
ov.utils.embedding(adata_new,
                   basis='X_mde',
                   color=["majority_voting"],
                   palette=sc.pl.palettes.default_102,
                   show=True,frameon='small',wspace=0.45)

In [ ]:
adata.obs['cell_types'] = adata_new.obs['majority_voting'].values

In [ ]:
adata.uns['cell_types_colors'] = adata_new.uns['majority_voting_colors']

In [ ]:
adata

In [ ]:
fig,ax=plt.subplots(figsize = (3,4))
ov.pl.cellproportion(adata=adata,celltype_clusters='cell_types',
                    groupby=['condition'],legend=True,ax=ax)

In [ ]:
adata_new.obs['majority_voting'].value_counts()

In [ ]:
adata

In [ ]:
ov.pl.violin(adata,keys='n_genes',groupby='cell_types',figsize=(12,3),
            rotation=90)

In [ ]:
palette = {
    "CA1-ProS": "#E2A528",
    "Oligo": "#6FA3C6",
    "Unassigned": "#7F7F7F"
}
ov.pl.embedding(adata,
                basis='X_mde',
                color= 'cell_types',
                size = 50,
                cmap= 'Greys',
                palette = palette,
                alpha = 0.8, 
                frameon='small')

In [ ]:
import os

plt.figure(figsize=(10, 9))


name = "cell_types"
fig = ov.pl.embedding(
    adata,
    basis="X_mde",
    color='cell_types',
    palette=palette,
    # color_map = cmap_custom,
    size=100,
    alpha=0.7,
    frameon="small",
    legend_loc="right margin",
    legend_fontsize=10,
    legend_fontweight="normal",
    add_outline=False,
    show=False,
    return_fig=True,
    vmax=6.0
)

save_dir = r"F:\Figures\PlaceCells\Figure3"
os.makedirs(save_dir, exist_ok=True)

fig.savefig(f"{save_dir}/{name}.svg", bbox_inches="tight")
fig.savefig(f"{save_dir}/{name}.png", dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
sc.tl.dendrogram(adata,'cell_types',use_rep='X_pca_harmony')
sc.tl.rank_genes_groups(adata, 'cell_types', use_rep='X_pca_harmony',
                        method='t-test',use_raw=False,key_added='celltypes_ttest')
ov.pl.rank_genes_groups_dotplot(adata,groupby='cell_types',
                                cmap='Spectral_r',key='celltypes_ttest',
                                standard_scale='var',n_genes=10,dendrogram=False)

In [ ]:
adata = adata[adata.obs["cell_types"] != "Unassigned"].copy()

In [ ]:
# 1) sync obs
adata_raw.obs = adata.obs.copy()

# 2) sync obsm
for key in adata.obsm.keys():
    adata_raw.obsm[key] = adata.obsm[key].copy()

# 3) sync uns（所有非基因相关的 uns 都可以同步）
for key in adata.uns.keys():
    # 跳过与 var /基因相关的 uns
    if key not in ["hvg", "pca", "scaled|original|pca_var_ratios", "scaled|original|cum_sum_eigenvalues"]:
        adata_raw.uns[key] = adata.uns[key].copy()

# 4) sync neighbor graph (非常重要)
adata_raw.obsp["distances"] = adata.obsp["distances"].copy()
adata_raw.obsp["connectivities"] = adata.obsp["connectivities"].copy()

# 5) sync neighbors metadata
adata_raw.uns["neighbors"] = adata.uns["neighbors"].copy()

print("obs, obsm, uns, obsp successfully synced!")


In [ ]:
print(adata_raw.layers.keys())
print(adata_raw.X[:10, :10])
print(adata_raw.layers["counts"][:10, :10])
print(np.max(adata_raw.X))

In [ ]:
# 检查是否为整数矩阵（允许 float dtype）
is_int = np.allclose(adata_raw.X.data, np.round(adata_raw.X.data))
print("X looks like integer UMI matrix:", is_int)

print("Max in X:", np.max(adata_raw.X))
print("Max in layers['counts']:", np.max(adata_raw.layers['counts']))

# 检查 X 和 counts 是否完全一致
same_matrix = np.allclose(
    adata_raw.X.toarray()[:200,:200],
    adata_raw.layers["counts"].toarray()[:200,:200]
)
print("X and layers['counts'] match in sample block:", same_matrix)


In [ ]:
import os
from datetime import date

file_path = r"F:\Code\codex\PlaceCell_snRNAseq\Data\FilteredData"
today = date.today().strftime("%Y%m%d")      
output_path = os.path.join(file_path, f"All_QC_filtered_annotation_{today}.h5ad") 

adata.write_h5ad(output_path, compression="gzip")

print(f"Successful saving：{output_path}")

In [ ]:
adata